# Full Train: RepViT-M1.5 + T5-Efficient-Mini on Kaggle T4 x2

Use this notebook after `kaggle_smoke_debug.ipynb` passes. This notebook runs full data prep, full feature cache, 6-epoch align, 6-epoch finetune, full evaluation, and benchmark.

In [ ]:
GITHUB_REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
GITHUB_BRANCH = "huy"
USE_SAFE_CONFIG = False

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
os.environ["USE_SAFE_CONFIG"] = "1" if USE_SAFE_CONFIG else "0"
print({"repo": GITHUB_REPO_URL, "branch": GITHUB_BRANCH, "safe_config": USE_SAFE_CONFIG})

In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
fi
cd Efficient_VLM_For_Autonomous_Driving
git pull --ff-only
python -m pip install -e .

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print({"hf_token_present": bool(os.environ.get("HF_TOKEN"))})

In [ ]:
import torch
count = torch.cuda.device_count()
print({"cuda": torch.cuda.is_available(), "gpu_count": count})
for idx in range(count):
    print(idx, torch.cuda.get_device_name(idx))
if count < 2:
    raise RuntimeError("Full train notebook requires Kaggle GPU T4 x2.")

In [ ]:
%%bash
set -e
nvidia-smi
ps -ef | grep -E "accelerate|efficient_vlm_ad|python" | grep -v grep || true

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
fi
python -m efficient_vlm_ad inspect-data --config "$CONFIG"
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset full
python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset full

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu
fi
python -m efficient_vlm_ad diagnose-train --config "$CONFIG" --stage align --debug --debug-samples 1
if [ -f "$PROFILE/checkpoints/align_latest.pt" ]; then
  echo "Skipping 20-step probe because align_latest.pt already exists."
else
  accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align --max-steps 20 --debug --debug-samples 1
  rm -f "$PROFILE/checkpoints/align_latest.pt" "$PROFILE/checkpoints/align_best.pt"
fi

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu
fi
RESUME=""
if [ -f "$PROFILE/checkpoints/align_latest.pt" ]; then
  RESUME="--resume $PROFILE/checkpoints/align_latest.pt"
fi
accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align $RESUME --debug --debug-samples 1

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu
fi
if [ -f "$PROFILE/checkpoints/finetune_latest.pt" ]; then
  RESUME="$PROFILE/checkpoints/finetune_latest.pt"
else
  RESUME="$PROFILE/checkpoints/align_best.pt"
fi
accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage finetune --resume "$RESUME" --debug --debug-samples 1

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_safe.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_safe
else
  CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu.yaml
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu
fi
CHECKPOINT="$PROFILE/checkpoints/best_model.pt"
if [ ! -f "$CHECKPOINT" ]; then
  CHECKPOINT="$PROFILE/checkpoints/finetune_best.pt"
fi
python -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint "$CHECKPOINT"
python -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint "$CHECKPOINT"

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_safe; else PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu; fi
ls -lh "$PROFILE/checkpoints"
cat "$PROFILE/metrics.json"
cat "$PROFILE/benchmark.json"